# Lesson 9.3 — Application Migration: AWS MGN, DataSync & Snow Family

**Level: Advanced** | **Exam:** SAP-C02 Domain 4, Task 4.2

## Application Migration Tools

### AWS Application Migration Service (MGN)
- Replaces Server Migration Service (SMS) — AWS recommends MGN for server migrations
- **Continuous block-level replication** of servers to AWS
- Cutover: launch replicated instance in AWS → test → promote
- Works for: physical servers, VMs (VMware, Hyper-V), other clouds

```
On-prem server → AWS Replication Agent → MGN → Staging EC2 → Cutover EC2
```

### AWS DataSync
- Automated data transfer between: on-prem ↔ S3, EFS, FSx
- Also: S3 ↔ EFS, S3 ↔ FSx (within AWS)
- Uses TLS encryption in transit
- Up to 10x faster than open-source tools (parallel multi-threaded transfers)
- Runs on DataSync Agent (deployed on-prem as VM)

### AWS Snow Family (Physical Data Transfer)
When internet transfer is impractical (petabyte scale or poor connectivity):

| Device | Storage | Transfer Speed | Use Case |
|--------|---------|---------------|---------|
| **Snowcone** | 8-14 TB | Low | Edge computing, tactical edge |
| **Snowball Edge (Storage)** | 80 TB | 1 Gbps | Large-scale migration |
| **Snowball Edge (Compute)** | 42 TB + GPU | 1 Gbps | ML at edge, compute + storage |
| **Snowmobile** | 100 PB | 10 Gbps | Exabyte-scale migrations |

### Rule of Thumb: Online vs Physical Transfer
```
1 TB at 100 Mbps internet → 1.3 days
10 TB at 100 Mbps → 13 days (use DataSync)
100 TB at 100 Mbps → 128 days → Use Snowball!
1 PB → Use Snowmobile
```

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Migration readiness assessment
import boto3

mgn = boto3.client('mgn', region_name='us-east-1')

try:
    # List source servers managed by MGN
    servers = mgn.describe_source_servers()['items']
    print(f"MGN Source Servers ({len(servers)}):")
    for server in servers:
        state = server.get('dataReplicationInfo', {}).get('dataReplicationState', 'Unknown')
        hostname = server.get('sourceProperties', {}).get('identificationHints', {}).get('hostname', 'Unknown')
        print(f"  {hostname:40} Replication: {state}")
    if not servers:
        print("  No source servers registered.")
        print("  To start: install AWS Replication Agent on on-prem servers")
except Exception as e:
    print(f"MGN: {e}")

# Data transfer calculator
print("\nData Transfer Method Selection Calculator:")
print(f"{'Data Size':15} {'Transfer Time at 100Mbps':30} {'Recommended Method'}")
print("-" * 75)
import math

data_sizes = [1, 10, 100, 500, 1000, 5000, 100000]  # GB
for size_gb in data_sizes:
    days = (size_gb * 8 * 1024) / (100 * 1024 * 1024 * 86400 / 1000000)  # 100 Mbps in Gbps
    if size_gb < 100:
        method = "Direct Upload (AWS CLI/SDK)"
    elif size_gb < 1000:
        method = "AWS DataSync"
    elif size_gb < 100000:
        method = "Snowball Edge"
    else:
        method = "Snowmobile"
    size_label = f"{size_gb}GB" if size_gb < 1000 else f"{size_gb//1000}TB"
    print(f"{size_label:15} {days:.1f} days{' ':25} {method}")

## Exercise

In [ ]:
# TODO: Plan a migration for a company with 500TB of NFS data on-premises
# They have a 1 Gbps internet connection (shared with production traffic)
# Peak hours: limited to 200 Mbps for migration
# They need the migration completed within 30 days
#
# Calculate which Snow Family device to use and create a migration timeline

data_size_tb = 500
internet_bandwidth_mbps = 200  # Available for migration
days_available = 30

# TODO: Calculate transfer time over internet
# TODO: Determine if Snow Family is needed
# TODO: If Snow Family: which device? How many?
# TODO: Create a timeline

import math

# Calculation hints:
# Transfer time (days) = (data_size_gb * 8 * 1024) / (bandwidth_mbps * 1000 * 86400 / 1000000)
# Snowball Edge: 80 TB usable, ~10 days shipping + loading

print(f"Data to migrate: {data_size_tb} TB")
print(f"Available bandwidth: {internet_bandwidth_mbps} Mbps")
print(f"Days available: {days_available}")
print()
# YOUR CALCULATIONS AND RECOMMENDATION HERE

## Summary
- **MGN (Application Migration Service)**: Continuous replication → minimal downtime for server migrations
- **DataSync**: Automated, accelerated file transfer for NFS/SMB → S3/EFS/FSx
- **Snow Family**: Physical transfer when internet would take too long or is unavailable
- Rule: < 1 TB → internet; 1-100 TB → DataSync; > 100 TB → Snowball Edge; > 10 PB → Snowmobile
- Always estimate transfer time: (data_GB × 8000) / (bandwidth_Mbps) / 86400 = days

---
**Next:** [Lesson 9.4 — Modernization: Serverless, Containers & Purpose-Built Databases](lesson-9.4-modernization.ipynb)